# Lab 02 — Bag-of-Words, TF-IDF and a text classifier

**Course:** Fundamentals of NLP
**Author:** Aymen Ben Brik

## Objectives

1. Build a Bag-of-Words document-term matrix from scratch.
2. Compute TF-IDF weights and contrast them with raw counts.
3. Train a logistic-regression text classifier on TF-IDF features.
4. Inspect the most discriminative features per class.
5. Compare classifier performance against a Naive Bayes baseline.

## Prerequisites

```bash
pip install scikit-learn matplotlib
```

## 1. Setup and dataset

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

# 4 well-separated newsgroup categories
categories = ['rec.sport.hockey', 'sci.space', 'comp.graphics', 'talk.politics.misc']
data = fetch_20newsgroups(subset='all', categories=categories,
                           remove=('headers', 'footers', 'quotes'))
print(f'Number of documents: {len(data.data)}')
print(f'Categories: {data.target_names}')
print(f'\nExample (label = {data.target_names[data.target[0]]}):')
print(data.data[0][:300], '...')

## 2. Bag-of-Words from scratch

In [ ]:
def bow_from_scratch(corpus, vocab=None):
    """Returns (matrix, vocab) where matrix[i, j] = count of vocab[j] in corpus[i]."""
    tokenized = [doc.lower().split() for doc in corpus]
    if vocab is None:
        all_words = sorted({w for doc in tokenized for w in doc})
        vocab = {w: j for j, w in enumerate(all_words)}
    matrix = np.zeros((len(corpus), len(vocab)), dtype=int)
    for i, doc in enumerate(tokenized):
        counts = Counter(doc)
        for w, c in counts.items():
            if w in vocab:
                matrix[i, vocab[w]] = c
    return matrix, vocab

small_corpus = [
    'the room was clean and quiet',
    'the breakfast was great and the room was clean',
    'service was slow but rooms were clean',
]
M, V = bow_from_scratch(small_corpus)
print('Vocabulary size:', len(V))
print('Matrix shape:', M.shape)
print('Document-term matrix:')
print(M)

## 3. CountVectorizer (sklearn equivalent)

In [ ]:
cv = CountVectorizer().fit(small_corpus)
X = cv.transform(small_corpus).toarray()
print('sklearn vocabulary:', cv.get_feature_names_out())
print('sklearn matrix:')
print(X)

## 4. TF-IDF: from raw counts to weighted features

In [ ]:
tfidf = TfidfVectorizer().fit(small_corpus)
X_tfidf = tfidf.transform(small_corpus).toarray().round(3)
print('Vocabulary:', tfidf.get_feature_names_out())
print('TF-IDF matrix:')
print(X_tfidf)

print('\nIDF of each word:')
for word, idf in zip(tfidf.get_feature_names_out(), tfidf.idf_):
    print(f'  {word:<12} IDF = {idf:.3f}')

**Observation.** Words appearing in every document (`the`, `was`) get the lowest IDF and contribute little to the representation. Discriminative words (`breakfast`, `service`, `slow`) get high IDF and dominate.

## 5. Train a TF-IDF + Logistic Regression classifier on 20 Newsgroups

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.2, random_state=42, stratify=data.target,
)

vectorizer = TfidfVectorizer(min_df=5, max_df=0.7, stop_words='english', ngram_range=(1, 2))
Xtr = vectorizer.fit_transform(X_train)
Xte = vectorizer.transform(X_test)
print(f'Train shape: {Xtr.shape}, Test shape: {Xte.shape}')

clf = LogisticRegression(max_iter=2000, C=1.0)
clf.fit(Xtr, y_train)
y_pred = clf.predict(Xte)
print(classification_report(y_test, y_pred, target_names=data.target_names))

## 6. Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(4)); ax.set_yticks(range(4))
ax.set_xticklabels(data.target_names, rotation=30, ha='right')
ax.set_yticklabels(data.target_names)
for i in range(4):
    for j in range(4):
        ax.text(j, i, cm[i, j], ha='center', va='center', color='black' if cm[i, j] < cm.max()/2 else 'white')
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Confusion matrix')
fig.colorbar(im); plt.tight_layout()
plt.show()

## 7. Most discriminative features per class

In [ ]:
feature_names = vectorizer.get_feature_names_out()
for class_idx, class_name in enumerate(data.target_names):
    top = np.argsort(clf.coef_[class_idx])[-10:][::-1]
    words = [feature_names[i] for i in top]
    print(f'\n{class_name}: {words}')

## 8. Naive Bayes baseline

In [ ]:
nb = MultinomialNB().fit(Xtr, y_train)
y_pred_nb = nb.predict(Xte)
print(classification_report(y_test, y_pred_nb, target_names=data.target_names))

**Comparison.** Naive Bayes is faster but typically a few F1 points behind logistic regression on TF-IDF features. Both are strong, fast baselines that should be tried before any deep model.

## Exercises

**Exercise 1.** Re-train the classifier without `stop_words='english'`. Does accuracy go up or down? Inspect the most discriminative words: do you see stop words appearing? Why or why not?

**Exercise 2.** Compute the TF-IDF representation by hand for the corpus:
- *d1: 'NLP is fun and useful'*
- *d2: 'Tunisia is a country in North Africa'*
- *d3: 'NLP is widely used in Tunisia'*

**Exercise 3.** Try `ngram_range=(1, 3)` and report the change in F1. What is the cost (vocabulary size, training time)?

**Exercise 4.** The classifier confuses some categories more than others. Identify the two most-confused classes from the confusion matrix and propose a hypothesis for why.

**Exercise 5 (synthesis).** Replace the categories with `['talk.politics.misc', 'talk.politics.guns', 'talk.politics.mideast']` (more semantically related). Does accuracy drop? Explain in terms of the limitations of bag-of-words.